In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
!pip install scikit-learn pandas Pillow

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
DRIVE_DIR  = '/content/drive/MyDrive'
IMAGES_DIR = '/content/drive/MyDrive/HAM10000'
CSV_PATH   = '/content/drive/MyDrive/HAM10000/HAM10000_metadata.csv'
SAVE_PATH  = '/content/drive/MyDrive/weights/resnet50_ham10000.pth'
os.makedirs('/content/drive/MyDrive/weights', exist_ok=True)
print('Paths set')

Paths set


In [6]:
import pandas as pd
df = pd.read_csv(CSV_PATH)
print(f'Records: {len(df)}')
print(df['dx'].value_counts())
print(f'Images found: {len(os.listdir(IMAGES_DIR))}')

Records: 10015
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
Images found: 10016


In [7]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [8]:
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [9]:
class HAMDataset(Dataset):
    def __init__(self, df, transform, le):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.le = le

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{IMAGES_DIR}/{row['image_id']}.jpg").convert('RGB')
        label = self.le.transform([row['dx']])[0]
        return self.transform(img), torch.tensor(label, dtype=torch.long)

In [10]:
le = LabelEncoder()
le.fit(df['dx'])
print('Classes:', list(le.classes_))

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['dx'], random_state=42)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

train_loader = DataLoader(HAMDataset(train_df, train_tf, le), batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(HAMDataset(val_df,   val_tf,   le), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print('Loaders ready')

Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
Train: 8012 | Val: 2003
Loaders ready


In [11]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 7)
model = model.to(DEVICE)
print(f'Model ready on {DEVICE}')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s]


Model ready on cuda


In [12]:
class_counts  = df['dx'].value_counts()
class_weights = torch.tensor(
    [1.0 / class_counts[cls] for cls in le.classes_],
    dtype=torch.float
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
print('Loss, optimizer, scheduler ready')

Loss, optimizer, scheduler ready


In [13]:
EPOCHS = 20
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):

    # Train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * images.size(0)
        _, predicted   = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()
        train_total   += images.size(0)

    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total   += images.size(0)

    train_acc = 100.0 * train_correct / train_total
    val_acc   = 100.0 * val_correct   / val_total
    scheduler.step()

    print(f'Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss/train_total:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss/val_total:.4f} Acc: {val_acc:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  Saved best model -> val acc: {val_acc:.2f}%')

print(f'\nDone. Best val accuracy: {best_val_acc:.2f}%')

Epoch 01/20 | Train Loss: 1.3231 Acc: 58.90% | Val Loss: 0.8256 Acc: 66.85%
  Saved best model -> val acc: 66.85%
Epoch 02/20 | Train Loss: 0.7892 Acc: 70.01% | Val Loss: 0.6955 Acc: 74.14%
  Saved best model -> val acc: 74.14%
Epoch 03/20 | Train Loss: 0.6200 Acc: 73.45% | Val Loss: 0.7975 Acc: 75.59%
  Saved best model -> val acc: 75.59%
Epoch 04/20 | Train Loss: 0.4882 Acc: 76.88% | Val Loss: 0.6182 Acc: 79.18%
  Saved best model -> val acc: 79.18%
Epoch 05/20 | Train Loss: 0.4852 Acc: 78.54% | Val Loss: 0.5451 Acc: 78.28%
Epoch 06/20 | Train Loss: 0.3871 Acc: 81.39% | Val Loss: 0.6242 Acc: 80.13%
  Saved best model -> val acc: 80.13%
Epoch 07/20 | Train Loss: 0.3339 Acc: 82.38% | Val Loss: 0.6089 Acc: 77.83%
Epoch 08/20 | Train Loss: 0.3185 Acc: 84.30% | Val Loss: 0.4656 Acc: 84.37%
  Saved best model -> val acc: 84.37%
Epoch 09/20 | Train Loss: 0.2501 Acc: 86.22% | Val Loss: 0.5339 Acc: 85.42%
  Saved best model -> val acc: 85.42%
Epoch 10/20 | Train Loss: 0.2117 Acc: 88.11% | Val

In [14]:
checkpoint = torch.load(SAVE_PATH, map_location='cpu')
print(f'Weights verified — {len(checkpoint)} layers saved')
print(f'File location: {SAVE_PATH}')

Weights verified — 320 layers saved
File location: /content/drive/MyDrive/weights/resnet50_ham10000.pth
